# ERBS: Basic Usage Guide

This notebook demonstrates how to set up an enhanced sampling simulation using the **Enhanced Representation-Based Sampling (ERBS)** library. ERBS accelerates sampling along slow collective variables (CVs) identified automatically from atomic descriptors.

## 1. System Setup
First, we define a simple system (a water molecule) and a base calculator (EMT).

In [ ]:
from ase.build import molecule
from ase.calculators.emt import EMT
from ase import units

atoms = molecule("H2O")
atoms.center(vacuum=4.0)
atoms.calc = EMT()

## 2. Configure ERBS

To use ERBS, we need to provide:
1. **Dimensionality Reduction**: A strategy to project high-dimensional descriptors to a few CVs (e.g., `GlobalPCA`).
2. **Energy Function Factory**: A strategy for the bias potential (e.g., `OPESExploreFactory`).
3. **Feature Builder**: A way to instantiate the descriptor (e.g., `DescriptorBuilder` for the built-in Gaussian Moments).

In [ ]:
from erbs.interfaces.ase import ERBS
from erbs.biases import OPESExploreFactory
from erbs.dim_reductions import GlobalPCA
from erbs.descriptors.factory import DescriptorBuilder

# Setup PCA to reduce to 2 components
pca = GlobalPCA(n_components=2)

# Setup OPES-Explore bias potential
energy_fn_factory = OPESExploreFactory(
    T=300, 
    dE=10 * units.kB * 300, # Expected barrier height
    a=0.5 # Bandwidth for kernels
)

# Use the built-in Gaussian Moment descriptor with 5 basis functions
feature_builder = DescriptorBuilder(n_basis=5, r_max=4.0)

# Instantiate the ERBS calculator
calc = ERBS(
    base_calc=atoms.calc,
    dim_reduction_factory=pca,
    energy_fn_factory=energy_fn_factory,
    feature_builder=feature_builder,
    interval=10, # Update bias every 10 steps
    update_iterations=100 # After 100 iterations, stop fitting PCA and only update kernels
)

atoms.calc = calc

## 3. Run Molecular Dynamics
Now we can run MD as usual using ASE's dynamics modules.

In [ ]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.verlet import VelocityVerlet
import numpy as np

MaxwellBoltzmannDistribution(atoms, temperature_K=300)
dyn = VelocityVerlet(atoms, timestep=0.5 * units.fs)

print(f"Initial energy: {atoms.get_potential_energy():.4f} eV")

energies = []
def log_energy():
    energies.append(atoms.get_potential_energy())

dyn.attach(log_energy, interval=1)
dyn.run(100)

print(f"Final energy: {atoms.get_potential_energy():.4f} eV")

## 4. Advanced: Custom Feature Functions

You can easily supply your own feature function using the `CustomBuilder`. This is useful if you want to use descriptors not natively supported by ERBS.

In [ ]:
from erbs.descriptors.factory import CustomBuilder
import jax.numpy as jnp
import jax

def my_simple_distance_builder(displacement_fn, box):
    """
    A custom factory that returns a simple distance-based descriptor.
    The displacement_fn is provided by the ERBS calculator.
    """
    # Precompute any JAX-MD metric functions if needed
    from apax.utils.jax_md_reduced import space
    metric = space.canonicalize_displacement_or_metric(displacement_fn)
    metric = space.map_bond(metric)
    
    def feature_fn(positions, numbers, idx, box, offsets):
        # Simple example: just count the number of neighbors as a 'feature'
        # In practice, you would use positions and idx to compute descriptors
        dr = metric(positions[idx[0]], positions[idx[1]], box=box)
        # Return shape: (n_atoms, n_features)
        # Here we just return dummy features for demonstration
        n_atoms = positions.shape[0]
        return jnp.ones((n_atoms, 2)) 
        
    return feature_fn

custom_builder = CustomBuilder(my_simple_distance_builder)

# You can now use this builder in the ERBS calculator
# calc = ERBS(..., feature_builder=custom_builder)